<a href="https://colab.research.google.com/github/EMej34/das172-examen2-Edwin-Reyes./blob/main/Script_principal_de_ejecuci%C3%B3n.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
Script principal de ejecución.py

"""

from aerocargo_matrix import (
    validar_matrices,
    calcular_ocupacion_sobrecarga,
    evaluar_balance,
    extraer_submatriz_critica,
)


def imprimir_matriz(matriz, titulo: str, unidad: str = "") -> None:
    print(f"\n{titulo}")
    print("-" * len(titulo))
    for fila in matriz:
        valores = "  ".join(f"{v:8.2f}{unidad}" for v in fila)
        print(f"[ {valores} ]")


def imprimir_encabezado(texto: str) -> None:
    print("\n" + "=" * 70)
    print(texto)
    print("=" * 70)


def main() -> None:
    # Matriz de cargas reales (N=4 filas x M=5 columnas) en kg.
    # Fila 0: proa | Fila 3: popa | Columna 0: babor | Columna 4: estribor
    cargas_reales = [
        [420, 380, 300, 410, 600],
        [500, 340, 280, 390, 410],
        [610, 420, 260, 300, 300],
        [300, 250, 240, 220, 200],
    ]

    # Matriz de capacidades máximas por celda, en kg.
    capacidades_maximas = [
        [500, 500, 500, 500, 500],
        [500, 500, 500, 500, 500],
        [500, 500, 500, 500, 500],
        [500, 500, 500, 500, 500],
    ]

    tolerancia_desbalance_kg = 150.0

    imprimir_encabezado("AEROCARGO-MATRIX — AUDITORÍA DE DISTRIBUCIÓN DE CARGA")

    imprimir_matriz(cargas_reales, "Matriz de Cargas Reales [kg]")
    imprimir_matriz(capacidades_maximas, "Matriz de Capacidades Máximas [kg]")

    # 1. Validación dimensional
    imprimir_encabezado("1. VALIDACIÓN DIMENSIONAL")
    es_valida = validar_matrices(cargas_reales, capacidades_maximas)
    print(f"Resultado de validación: {'APROBADO' if es_valida else 'RECHAZADO'}")

    if not es_valida:
        print("Las matrices no son válidas. Se detiene la ejecución.")
        return

    # 2. Ocupación y sobrecarga
    imprimir_encabezado("2. OCUPACIÓN Y DETECCIÓN DE SOBRECARGA")
    resultado_ocupacion = calcular_ocupacion_sobrecarga(cargas_reales, capacidades_maximas)
    imprimir_matriz(resultado_ocupacion["matriz_porcentajes"], "Matriz de Porcentaje de Ocupación", "%")

    celdas_sobrecargadas = resultado_ocupacion["celdas_sobrecargadas"]
    if celdas_sobrecargadas:
        print(f"\nCeldas en sobrecarga crítica (> 100.0%): {len(celdas_sobrecargadas)}")
        for (fila, columna) in celdas_sobrecargadas:
            porcentaje = resultado_ocupacion["matriz_porcentajes"][fila][columna]
            print(f"  - Celda (fila={fila}, columna={columna}): {porcentaje:.2f}%")
    else:
        print("\nNo se detectaron celdas en sobrecarga.")

    # 3. Balance lateral y longitudinal
    imprimir_encabezado("3. BALANCE LATERAL Y LONGITUDINAL")
    resultado_balance = evaluar_balance(cargas_reales, tolerancia_desbalance_kg)

    print("Peso total por fila longitudinal [kg]:")
    for indice, peso in enumerate(resultado_balance["pesos_fila"]):
        print(f"  Fila {indice}: {peso:.2f} kg")

    print(f"\nDesbalance lateral: {resultado_balance['desbalance_lateral']:.2f} kg")
    print(f"Tolerancia permitida: {tolerancia_desbalance_kg:.2f} kg")
    estado_balance = "APROBADO" if resultado_balance["balance_ok"] else "RECHAZADO"
    print(f"Estado de balance: {estado_balance}")

    # 4. Submatriz de sobrecarga crítica
    imprimir_encabezado("4. EXTRACCIÓN DE SUBMATRIZ DE SOBRECARGA CRÍTICA")
    k, p = 2, 2
    submatriz_critica = extraer_submatriz_critica(
        resultado_ocupacion["matriz_porcentajes"], k, p
    )
    imprimir_matriz(submatriz_critica, f"Zona crítica ({k}x{p}) con mayor ocupación promedio", "%")


if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'aerocargo_matrix'